In [1]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "file:///mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/mlruns"

import mlflow
print(mlflow.get_tracking_uri())  

/mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


file:///mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/mlruns


In [2]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [3]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

/mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/.venv/bin/python
3.2.0
/mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/.venv/lib/python3.12/site-packages/xgboost/__init__.py


In [4]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [5]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"/mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv(r"/mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [6]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [7]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
#mlflow.set_tracking_uri("file:///mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/mlruns")
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

/mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/26 08:12:11 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-04-26 08:12:11,090] A new study created in memory with name: no-name-867b75af-63e1-4d05-9a30-befae6471bab
[I 2026-04-26 08:13:12,685] Trial 0 finished with value: 74923.82597343744 and parameters: {'n_estimators': 226, 'max_depth': 7, 'learning_rate': 0.07764859996169739, 'subsample': 0.9613979094898859, 'colsample_by

Best params: {'n_estimators': 700, 'max_depth': 10, 'learning_rate': 0.032082288072628445, 'subsample': 0.6163659798818756, 'colsample_bytree': 0.6202164745789467, 'min_child_weight': 10, 'gamma': 4.984371507005888, 'reg_alpha': 6.491628049411741e-06, 'reg_lambda': 1.6600561149187007e-05}


In [8]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    print(mlflow.get_tracking_uri())
    print(mlflow.get_artifact_uri())
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 30603.422165776145
RMSE: 69659.6256848458
R²: 0.9625007739752057
file:///mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/mlruns
file:///mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/mlruns/630691151906002186/33ca2466de2d4ab88ad9bdc6d7e40f70/artifacts
